# Step 01 — Data Inventory & Strategy

Explores the InHARD dataset clip counts, class imbalance, subject distribution, and
validates the 3-view frame extraction before committing to a long embedding run.

In [ ]:
import sys
from pathlib import Path
NB_DIR = Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import cv2
from IPython.display import display

## 1a. Full Dataset Inventory

In [ ]:
from lib.inhard import analyze_training_clips, label_counts, subject_counts
from lib.paths import find_inhard_root
from lib.constants import TRAINABLE_ACTIONS, BLOCKED_ACTIONS

root   = find_inhard_root()
report = analyze_training_clips(root, exclude_labels=BLOCKED_ACTIONS, clips_per_class=None)

if not report.ok:
    print(f"ERROR: {report.error}")
else:
    lc = label_counts(report.clips)
    sc = subject_counts(report.clips)
    print(f"Total trainable clips : {len(report.clips)}")
    print(f"Classes               : {report.n_classes}")
    print(f"Subjects              : {len(sc)}")
    print(f"Imbalance ratio       : {lc.max()/lc.min():.1f}x  (max={lc.max()} min={lc.min()})")
    print()
    display(lc.rename('clips').to_frame())

In [ ]:
# Class imbalance bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(lc)))
lc.sort_values().plot(kind='barh', ax=axes[0], color=colors)
axes[0].axvline(lc.mean(), color='navy', lw=2, ls='--', label=f'mean={lc.mean():.0f}')
axes[0].set_title('Clips per class — full InHARD (ALL clips)', fontsize=12)
axes[0].set_xlabel('clips')
axes[0].legend()
for bar, val in zip(axes[0].patches, lc.sort_values().values):
    axes[0].text(bar.get_width()+2, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=8)

sc.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Clips per subject', fontsize=12)
axes[1].set_xlabel('Subject')
axes[1].set_ylabel('clips')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('outputs/01_data_inventory.png', dpi=130, bbox_inches='tight')
plt.show()

## 1b. 3-View Frame Extraction Validation
Visually confirm sub-view extraction from InHARD composite frames.

In [ ]:
from lib.crop_extract import extract_all_views, frame_to_person_crop
from lib.embeddings import extract_clip_from_file
from lib.constants import INHARD_VIEW_TOPDOWN, INHARD_VIEW_SIDE, INHARD_VIEW_FRONT

# Pick one clip per class for visual inspection
sample_clips = {}
for rec in report.clips:
    if rec.label not in sample_clips:
        sample_clips[rec.label] = rec.path
    if len(sample_clips) == len(TRAINABLE_ACTIONS):
        break

# Show one example — 3 views + YOLO crop
example_label = list(sample_clips.keys())[0]
example_path  = sample_clips[example_label]
frames = extract_clip_from_file(example_path, max_frames=32)
mid    = frames[len(frames)//2] if frames else None

if mid is not None:
    views = extract_all_views(mid)
    crop, bbox = frame_to_person_crop(mid, view=INHARD_VIEW_TOPDOWN)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    titles = ['Full composite (1280×720)', 'Top-down (overhead)', 'Side (eye-level)', 'Front (low-angle)']
    imgs   = [mid, views[INHARD_VIEW_TOPDOWN], views[INHARD_VIEW_SIDE], views[INHARD_VIEW_FRONT]]
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    fig.suptitle(f'3-view extraction — class: {example_label}', fontsize=12)
    plt.tight_layout()
    plt.show()

    # Show YOLO detection on top-down
    td = cv2.cvtColor(views[INHARD_VIEW_TOPDOWN].copy(), cv2.COLOR_BGR2RGB)
    if bbox:
        x1,y1,x2,y2,conf = bbox
        import cv2 as cv
        cv.rectangle(td,(x1,y1),(x2,y2),(0,200,0),2)
        cv.putText(td,f'person {conf:.2f}',(x1,max(y1-5,15)),cv.FONT_HERSHEY_SIMPLEX,0.5,(0,200,0),1)
    fig2, ax2 = plt.subplots(1,2,figsize=(10,4))
    ax2[0].imshow(td); ax2[0].set_title('YOLO detection on top-down'); ax2[0].axis('off')
    ax2[1].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); ax2[1].set_title('Person crop (224×224)'); ax2[1].axis('off')
    plt.tight_layout(); plt.show()
    print(f'YOLO bbox: {bbox}')
else:
    print('Could not load frames — check InHARD path')

## 1c. Clip Duration Analysis

In [ ]:
import re

durations = []
for rec in report.clips:
    m = re.search(r'_(\d+\.\d+)_(\d+\.\d+)\.mp4$', rec.path.name)
    if m:
        dur = float(m.group(2)) - float(m.group(1))
        durations.append({'label': rec.label, 'duration': dur, 'subject': rec.subject})

df_dur = pd.DataFrame(durations)
if not df_dur.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    df_dur.groupby('label')['duration'].mean().sort_values().plot(kind='barh', ax=axes[0], color='coral')
    axes[0].set_title('Mean clip duration per class (seconds)')
    df_dur['duration'].hist(bins=30, ax=axes[1], color='steelblue', edgecolor='white')
    axes[1].axvline(df_dur['duration'].mean(), color='red', lw=2, ls='--', label=f"mean={df_dur['duration'].mean():.2f}s")
    axes[1].set_title('Duration distribution (all clips)')
    axes[1].set_xlabel('seconds')
    axes[1].legend()
    plt.tight_layout(); plt.show()
    print(df_dur['duration'].describe())

## 1d. Frame Strip — All Classes, Top-Down View

In [ ]:
from lib.har_analysis import plot_frame_strip_comparison
from lib.paths import OUTPUTS_DIR

# Group 3 clips per class
by_class = {}
for rec in report.clips:
    by_class.setdefault(rec.label, []).append(rec.path)

OUTPUTS_DIR.mkdir(exist_ok=True)
out = plot_frame_strip_comparison(by_class, OUTPUTS_DIR, n_clips=3, n_frames_per_clip=5, view='topdown')
from IPython.display import Image
display(Image(out, width=900))